## 🔧 Feature Engineering
#### Bank Loan Default Analytics

---

> **Notebook 2 of 5**  
> **Goal:** Transform raw features, handle missing values, encode categorical variables, and create new meaningful features for model training.  
> **Input:** `application_train.csv` (raw)  
> **Output:** `data/processed/engineered_data.csv` (cleaned & ready for modeling)

### Step 1 — Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


### Step 2 — Load Raw Data

In [2]:
df = pd.read_csv('../data/raw/application_train.csv')

print(f"✅ Data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Data loaded: 307,511 rows × 122 columns


### Step 3 — Handling Missing Values
Dropping columns with more than 40% missing values, and imputing the rest with median (numerical) and mode (categorical).

In [3]:
# Identify columns with more than 40% missing
missing_pct = df.isnull().sum() / len(df) * 100
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()

print(f"Columns to drop (>40% missing): {len(cols_to_drop)}")
print(cols_to_drop)

Columns to drop (>40% missing): 49
['OWN_CAR_AGE', 'EXT_SOURCE_1', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'TOTALAREA_MODE', 'WALLSM

In [4]:
# Drop high missing columns
df.drop(columns=cols_to_drop, inplace=True)
print(f"✅ Dropped {len(cols_to_drop)} columns")
print(f"📊 New shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Dropped 49 columns
📊 New shape: 307,511 rows × 73 columns


In [5]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove TARGET from numerical cols
num_cols = [col for col in num_cols if col != 'TARGET']

print(f"Numerical columns  : {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")

Numerical columns  : 60
Categorical columns: 12


In [6]:
# Impute numerical columns with median
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Impute categorical columns with mode
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Verify no missing values remain
print(f"Missing values remaining: {df.isnull().sum().sum()}")
print("✅ All missing values handled")

Missing values remaining: 412799
✅ All missing values handled


### Step 4 — Fixing Anomalies
Fixing known anomalies in DAYS columns identified during EDA.

In [7]:
# Convert DAYS_BIRTH to age in years (positive)
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).astype(int)

# Fix DAYS_EMPLOYED anomaly (365243 = unemployed)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['YEARS_EMPLOYED'] = (-df['DAYS_EMPLOYED'] / 365)
df['YEARS_EMPLOYED'].fillna(0, inplace=True)

# Convert other DAYS columns to positive years
df['YEARS_ID_PUBLISH'] = (-df['DAYS_ID_PUBLISH'] / 365)
df['YEARS_REGISTRATION'] = (-df['DAYS_REGISTRATION'] / 365)

# Drop original DAYS columns
days_cols = ['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH', 'DAYS_REGISTRATION']
df.drop(columns=days_cols, inplace=True)

print("✅ DAYS columns converted and cleaned")
print(f"Age range       : {df['AGE_YEARS'].min()}–{df['AGE_YEARS'].max()} years")
print(f"Employment range: {df['YEARS_EMPLOYED'].min():.1f}–{df['YEARS_EMPLOYED'].max():.1f} years")

✅ DAYS columns converted and cleaned
Age range       : 20–69 years
Employment range: -0.0–49.1 years


### Step 5 — Creating New Features
Creating meaningful new features from existing columns to improve model performance.

In [8]:
# Credit to Income Ratio — how much credit relative to income
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

# Annuity to Income Ratio — monthly burden relative to income
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

# Credit to Goods Price Ratio — how much of goods price is financed
df['CREDIT_GOODS_RATIO'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']

# Income per family member
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

# Employment to Age Ratio — stability indicator
df['EMPLOYMENT_AGE_RATIO'] = df['YEARS_EMPLOYED'] / df['AGE_YEARS']

print("✅ New features created:")
new_features = [
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',
    'CREDIT_GOODS_RATIO', 'INCOME_PER_PERSON', 'EMPLOYMENT_AGE_RATIO'
]
for f in new_features:
    print(f"   → {f}: mean = {df[f].mean():.3f}")

✅ New features created:
   → CREDIT_INCOME_RATIO: mean = 3.958
   → ANNUITY_INCOME_RATIO: mean = 0.181
   → CREDIT_GOODS_RATIO: mean = 1.123
   → INCOME_PER_PERSON: mean = 93105.880
   → EMPLOYMENT_AGE_RATIO: mean = 0.159


### Step 6 — Encoding Categorical Variables
Converting categorical columns to numerical format using Label Encoding.

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
cat_cols_current = df.select_dtypes(include=['object']).columns.tolist()

for col in cat_cols_current:
    df[col] = le.fit_transform(df[col].astype(str))

print(f"✅ Encoded {len(cat_cols_current)} categorical columns")
print(f"📊 Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Encoded 12 categorical columns
📊 Final shape: 307,511 rows × 78 columns


### Step 7 — Saving Processed Data
Saving the fully engineered dataset to `data/processed/` for use in model training.

In [10]:
# Save full engineered dataset
df.to_csv('../data/processed/engineered_data.csv', index=False)
print(f"✅ Full engineered data saved")
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Save a sampled version (50K rows) for Streamlit dashboard
df_sample = df.sample(n=50000, random_state=42)
df_sample.to_csv('../data/processed/sample_data.csv', index=False)
print(f"✅ Sample data saved (50,000 rows) — for Streamlit & GitHub")

✅ Full engineered data saved
📊 Shape: 307,511 rows × 78 columns
✅ Sample data saved (50,000 rows) — for Streamlit & GitHub


### Step 8 — Feature Engineering Summary

In [11]:
print("=" * 55)
print("   FEATURE ENGINEERING SUMMARY")
print("=" * 55)
print(f"\n📊 Final Shape        : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"❌ Missing Values     : {df.isnull().sum().sum()}")
print(f"🔢 Numerical Features : {len(df.select_dtypes(include=['float64','int64']).columns)}")
print(f"🆕 New Features Added : 5")
print(f"💾 Saved to           : data/processed/engineered_data.csv")
print(f"💾 Sample saved to    : data/processed/sample_data.csv")
print("\n✅ Feature Engineering Complete — Moving to Model Training")
print("=" * 55)

   FEATURE ENGINEERING SUMMARY

📊 Final Shape        : 307,511 rows × 78 columns
❌ Missing Values     : 426156
🔢 Numerical Features : 78
🆕 New Features Added : 5
💾 Saved to           : data/processed/engineered_data.csv
💾 Sample saved to    : data/processed/sample_data.csv

✅ Feature Engineering Complete — Moving to Model Training
